# Analisi emotiva di una discussione completa

Analizziamo ora una discussione Reddit (post + tutti i suoi commenti).

Si calcola prima l' **emozione del post** — con il metodo finale (ELIta α=0.5 + corpus_mean ItEm), poi l'**emozione di ogni commento** e infine l'**emozione complessiva della discussione**.

Di default viene selezionato il post con più commenti nel corpus.

## Import e configurazione

In [18]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from Fase3.support import ( BASIC_EMOTIONS, EMOTION_COLORS, POS_FILTER, load_corpus, load_elita_matrix, compute_mu_e,
                            score_single_document, normalizza, emozione_dominante, plot_emotion_bars, plot_radar_single, plot_radar_comparison,
)

CORPUS_CSV   = Path('corpus_Italia_multi.csv')
TOKENS_CSV   = Path('tokens_Italia_multi.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')

print('Configurazione caricata.')

Configurazione caricata.


## Caricamento dati

In [19]:
df_corpus, df_tokens = load_corpus(CORPUS_CSV, TOKENS_CSV)
df_elita = load_elita_matrix(ALPHA_05_CSV)

df_f     = df_tokens[df_tokens['pos'].isin(POS_FILTER)].copy()
tok_dict = df_f.groupby('doc_id')['lemma'].apply(list).to_dict()
elita_idx = set(df_elita.index)

mu_e = compute_mu_e(df_corpus, df_tokens, df_elita)

print('Corpus:', len(df_corpus))
print('Token:', len(df_tokens))
print('Post:', (df_corpus['type'] == 'post').sum())
print('Commenti:', (df_corpus['type'] == 'comment').sum())

Corpus: 4311
Token: 146186
Post: 284
Commenti: 4027


## Selezione del post

Di default: il post con più commenti nel corpus.

In [21]:
POST_ID = None   # None per il post con più commenti, altrimenti si può specificare un id esistente

df_comments_only = df_corpus[df_corpus['type'] == 'comment']
df_posts_only    = df_corpus[df_corpus['type'] == 'post']

if POST_ID is None:
    post_id_max = df_comments_only['post_id'].value_counts().idxmax() # Post con più commenti nel corpus
    POST_ID = post_id_max

post_row = df_posts_only[df_posts_only['doc_id'] == POST_ID].iloc[0]   # Riga del post

# Commenti del post
df_disc_comments = df_comments_only[df_comments_only['post_id'] == POST_ID].reset_index(drop=True)

print(f'Post selezionato : {POST_ID}')
print(f'Autore           : {post_row["author"]}')
print(f'Score Reddit     : {post_row["score"]}')
print(f'Commenti nel corpus: {len(df_disc_comments)}')
print()
print('Testo del post:')
print('=' * 70)
print(post_row['text'])
print('=' * 70)

Post selezionato : 1e4lf9i
Autore           : Technical_Ad_4299
Score Reddit     : 118
Commenti nel corpus: 50

Testo del post:
Perchè sui social nelle notizie sulla guerra in Ucraina quasi tutti i commenti sono filorussi?


## Analisi emotiva del post

In [23]:
sc_post_raw, found_post = score_single_document(POST_ID, tok_dict, elita_idx, df_elita)
sc_post_norm = normalizza(sc_post_raw, mu_e)
emo_post     = emozione_dominante(sc_post_norm)

print(f'Token usati per il post: {found_post}')

df_scores = pd.DataFrame([
    {'emozione': e, 'S_e (raw)': sc_post_raw[e], 'μ_e': mu_e[e], 'S_e_norm': sc_post_norm[e]}
    for e in BASIC_EMOTIONS
]).set_index('emozione').round(3)
df_scores['dominante'] = df_scores.index == emo_post
display(df_scores)

print(f'=> Emozione del POST: {emo_post.upper()}')

Token usati per il post: 3


,S_e (raw),μ_e,S_e_norm,dominante
emozione,,,,
gioia,0.653,6.070,0.108,False
aspettativa,1.134,7.226,0.157,False
rabbia,1.835,4.426,0.415,False
disgusto,1.387,3.126,0.444,True
tristezza,1.704,4.522,0.377,False
sorpresa,1.387,5.847,0.237,False
paura,1.797,5.006,0.359,False
fiducia,0.605,6.256,0.097,False


=> Emozione del POST: DISGUSTO


In [24]:
plot_radar_single(
    sc_post_norm, emo_post,
    title=f'Profilo emotivo del post {POST_ID} — emozione dominante: {emo_post.upper()}',
    height=450,
).show()

## Analisi emotiva di ogni commento

In [25]:
comment_results = []
for _, row in df_disc_comments.iterrows():
    sc_raw, found = score_single_document(row['doc_id'], tok_dict, elita_idx, df_elita)
    sc_norm = normalizza(sc_raw, mu_e)
    emo     = emozione_dominante(sc_norm)
    comment_results.append({
        'doc_id'         : row['doc_id'],
        'rank_by_score'  : row.get('rank_by_score', ''),
        'score_reddit'   : row['score'],
        'n_token_matched': found,
        'emozione'       : emo,
        'text'           : str(row['text']),
        **{f'norm_{e}': sc_norm[e] for e in BASIC_EMOTIONS}
    })

df_results = pd.DataFrame(comment_results)

print(f'Commenti analizzati: {len(df_results)}')
print(f'Con almeno 1 token in ELIta: {(df_results["n_token_matched"] > 0).sum()}')

Commenti analizzati: 50
Con almeno 1 token in ELIta: 49


In [26]:
print('Distribuzione emozioni dominanti:')
counts = df_results['emozione'].value_counts()
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts.get(e, 0)
    if n > 0:
        print(f'  {e:<15s} {n:>3d}  {"█" * n}')

Distribuzione emozioni dominanti:
  gioia             3  ███
  aspettativa       1  █
  rabbia            3  ███
  disgusto         24  ████████████████████████
  tristezza         8  ████████
  sorpresa          3  ███
  paura             3  ███
  fiducia           4  ████
  neutrale          1  █


In [27]:
display(df_results[['rank_by_score', 'score_reddit', 'emozione', 'n_token_matched', 'text']]
        .sort_values('rank_by_score').reset_index(drop=True))

,rank_by_score,score_reddit,emozione,n_token_matched,text
0,1.0,28,disgusto,11,"Siamo pieni di fascisti, ai fascisti viene dur..."
1,2.0,14,fiducia,25,L'appendere i pelati a essiccare a testa in gi...
2,3.0,12,disgusto,6,oddio se i piedi son troppo piccoli poi non sc...
3,4.0,7,disgusto,178,Penso anch'io che sia così. Siamo portati a no...
4,5.0,4,paura,7,Perché i bot hanno una cosa che ormai la gente...
5,6.0,4,disgusto,22,Ma non è così.\nNessuno vuole che un Putin all...
6,7.0,3,disgusto,17,Si hai ragione pure loro stan messi male. Alla...
7,8.0,3,disgusto,8,Non capisco il nesso tra questi argomenti. Ma ...
8,9.0,3,disgusto,14,A me sembra che piace tantissimo ai compagni. ...
9,10.0,3,tristezza,36,"Perché la gente seria, con la testa sulle spal..."


In [28]:
plot_emotion_bars(
    counts, len(df_results),
    title=f'Distribuzione emozioni dominanti — commenti del post {POST_ID}',
    height=400,
).show()

# Heatmap: ogni riga = un commento, colonne = emozioni normalizzate
norm_cols = [f'norm_{e}' for e in BASIC_EMOTIONS]
df_heat = df_results[norm_cols].copy()
df_heat.columns = BASIC_EMOTIONS

import plotly.express as px
px.imshow(
    df_heat.T,
    labels=dict(x='Commento (rank)', y='Emozione', color='Score norm.'),
    x=[f'#{i+1}' for i in range(len(df_results))],
    color_continuous_scale='YlOrRd',
    title=f'Heatmap score normalizzati per commento — post {POST_ID}',
    height=400,
).show()

## Emozione complessiva della discussione

L'emozione della **discussione** è calcolata come media dei vettori normalizzati di tutti i documenti (post + commenti).  
Mostriamo anche il confronto tra emozione del post e emozione media dei commenti.

In [30]:
# Vettori normalizzati: post + commenti
all_norm_vecs = [sc_post_norm] + [
    {e: row[f'norm_{e}'] for e in BASIC_EMOTIONS}
    for _, row in df_results.iterrows()
]
df_norm_vecs = pd.DataFrame(all_norm_vecs)

# Media commenti (senza post)
sc_commenti_mean = df_norm_vecs.iloc[1:][BASIC_EMOTIONS].mean().to_dict()
emo_commenti     = emozione_dominante(sc_commenti_mean)

# Media intera discussione (post + commenti)
sc_discussione  = df_norm_vecs[BASIC_EMOTIONS].mean().to_dict()
emo_discussione = emozione_dominante(sc_discussione)

print(f'Post:        {emo_post.upper()}')
print(f'Commenti:    {emo_commenti.upper()}')
print(f'Discussione: {emo_discussione.upper()}')

Post:        DISGUSTO
Commenti:    DISGUSTO
Discussione: DISGUSTO


In [31]:
df_riepilogo = pd.DataFrame([
    {'emozione': e, 'post': sc_post_norm[e], 'media commenti': sc_commenti_mean[e], 'discussione': sc_discussione[e]}
    for e in BASIC_EMOTIONS
]).set_index('emozione').round(3)
display(df_riepilogo)

,post,media commenti,discussione
emozione,,,
gioia,0.108,1.527,1.499
aspettativa,0.157,1.592,1.563
rabbia,0.415,1.913,1.883
disgusto,0.444,1.962,1.933
tristezza,0.377,1.870,1.841
sorpresa,0.237,1.657,1.629
paura,0.359,1.854,1.825
fiducia,0.097,1.597,1.568


In [32]:
plot_radar_comparison(
    traces_data=[
        ('Post',               sc_post_norm,      EMOTION_COLORS.get(emo_post, '#999'), 0.5, 'solid'),
        ('Media commenti',     sc_commenti_mean,  '#1E88E5',                            0.4, 'solid'),
        ('Discussione totale', sc_discussione,    '#43A047',                            0.3, 'dot'),
    ],
    title=f'Confronto emotivo: Post vs Commenti vs Discussione — {POST_ID}',
).show()

In [33]:
# Grafico a barre affiancate
fig2 = go.Figure()
for label, sc, color in [
    ('Post',               sc_post_norm,      '#FB8C00'),
    ('Media commenti',     sc_commenti_mean,  '#1E88E5'),
    ('Discussione totale', sc_discussione,    '#43A047'),
]:
    fig2.add_trace(go.Bar(
        name=label, x=BASIC_EMOTIONS,
        y=[sc[e] for e in BASIC_EMOTIONS],
        marker_color=color, opacity=0.85,
    ))
fig2.update_layout(
    barmode='group',
    title=f'Score normalizzati: Post vs Commenti vs Discussione — {POST_ID}',
    xaxis_title='Emozione', yaxis_title='Score normalizzato', height=450,
)
fig2.show()

## Accordo/disaccordo tra post e commenti

Verifichiamo se l'emozione del post coincide con quella prevalente nei commenti.

In [35]:
n_concordi   = (df_results['emozione'] == emo_post).sum()
n_discordi   = len(df_results) - n_concordi
perc_concordi = n_concordi / len(df_results) * 100 if len(df_results) > 0 else 0

print(f'Emozione del post              : {emo_post.upper()}')
print(f'Commenti con stessa emozione   : {n_concordi}/{len(df_results)} ({perc_concordi:.1f}%)')
print(f'Commenti con emozione diversa  : {n_discordi}/{len(df_results)} ({100-perc_concordi:.1f}%)')

Emozione del post              : DISGUSTO
Commenti con stessa emozione   : 24/50 (48.0%)
Commenti con emozione diversa  : 26/50 (52.0%)


In [37]:
# Distribuzione emozioni nei commenti che differiscono dal post
df_discordi = df_results[df_results['emozione'] != emo_post]
if len(df_discordi) > 0:
    print('Emozioni nei commenti discordi:')
    for e, cnt in df_discordi['emozione'].value_counts().items():
        print(f'  {e:<15s}: {cnt}')

Emozioni nei commenti discordi:
  tristezza      : 8
  fiducia        : 4
  paura          : 3
  rabbia         : 3
  gioia          : 3
  sorpresa       : 3
  neutrale       : 1
  aspettativa    : 1


In [38]:
fig = go.Figure(go.Pie(                             # Grafico accordo vs disaccordo
    labels=['Accordo con post', 'Emozione diversa'],
    values=[n_concordi, n_discordi],
    marker_colors=[EMOTION_COLORS.get(emo_post, '#FB8C00'), '#BDBDBD'],
    hole=0.4
))
fig.update_layout(
    title=f'Accordo emotivo commenti con post ({emo_post.upper()}) — {POST_ID}',
    height=400
)
fig.show()

## Shift emotivo dal post alla discussione (corpus completo)

Per ogni post nel corpus calcoliamo:
- **emozione del post** (metodo finale: ELIta α=0.5 + `corpus_mean`)
- **emozione della discussione** = emozione dominante del vettore medio dei commenti normalizzati

In [40]:
from Fase3.support import POSITIVE, NEGATIVE

MIN_COMMENTS = 2   # ignora post con meno di N commenti nel corpus

df_posts_only    = df_corpus[df_corpus['type'] == 'post']
df_comments_only = df_corpus[df_corpus['type'] == 'comment']

def polarita(emo):
    if emo in POSITIVE: return 'positiva'
    if emo in NEGATIVE: return 'negativa'
    return 'neutrale'

shift_rows = []     # Loop su tutti i post

for _, post_row in df_posts_only.iterrows():
    pid = post_row['doc_id']

    sc_post_raw, found_post = score_single_document(pid, tok_dict, elita_idx, df_elita)
    if found_post == 0:
        continue  # post senza copertura ELIta → skip

    sc_post_n = normalizza(sc_post_raw, mu_e)
    emo_post  = emozione_dominante(sc_post_n)

    comms = df_comments_only[df_comments_only['post_id'] == pid]
    if len(comms) < MIN_COMMENTS:
        continue

    # Vettore medio dei commenti (solo quelli coperti da ELIta)
    comm_vecs = []
    for _, c_row in comms.iterrows():
        sc_c_raw, found_c = score_single_document(c_row['doc_id'], tok_dict, elita_idx, df_elita)
        if found_c == 0:
            continue
        comm_vecs.append(normalizza(sc_c_raw, mu_e))

    if not comm_vecs:
        continue

    mean_disc = {e: sum(v[e] for v in comm_vecs) / len(comm_vecs) for e in BASIC_EMOTIONS}
    emo_disc  = emozione_dominante(mean_disc)

    shift_rows.append({
        'post_id'       : pid,
        'n_commenti'    : len(comms),
        'n_comm_scored' : len(comm_vecs),
        'emo_post'      : emo_post,
        'emo_disc'      : emo_disc,
        'pol_post'      : polarita(emo_post),
        'pol_disc'      : polarita(emo_disc),
        'shift'         : emo_post != emo_disc,
        'shift_pol'     : polarita(emo_post) != polarita(emo_disc),
    })

df_shift = pd.DataFrame(shift_rows)

print(f'Post analizzati     : {len(df_shift)}  (≥{MIN_COMMENTS} commenti + copertura ELIta)')
print(f'Shift emozione      : {df_shift["shift"].sum()} ({df_shift["shift"].mean()*100:.1f}%)')
print(f'Shift polarità      : {df_shift["shift_pol"].sum()} ({df_shift["shift_pol"].mean()*100:.1f}%)')

Post analizzati     : 184  (≥2 commenti + copertura ELIta)
Shift emozione      : 129 (70.1%)
Shift polarità      : 60 (32.6%)


Qui mostriamo quanti cambiamenti di emozione e polarità si osservano tra post e discussione, considerando solo i post con almeno 2 commenti nel corpus (e copertura ELIta).

In [41]:
print('Emozione POST:        ', df_shift['emo_post'].value_counts().to_dict())
print('Emozione DISCUSSIONE: ', df_shift['emo_disc'].value_counts().to_dict())

Emozione POST:         {'gioia': 52, 'sorpresa': 30, 'disgusto': 22, 'tristezza': 19, 'fiducia': 18, 'rabbia': 16, 'paura': 15, 'aspettativa': 12}
Emozione DISCUSSIONE:  {'gioia': 54, 'disgusto': 46, 'fiducia': 20, 'aspettativa': 18, 'sorpresa': 16, 'tristezza': 12, 'paura': 12, 'rabbia': 6}


Dei post analizzati, mostriamo la distribuzione delle emozioni del post e della discussione.


Poi costruiamo la **matrice di transizione** **post\_emotion** -> **discussion\_emotion**.

In [43]:
emos_order = [e for e in BASIC_EMOTIONS + ['neutrale']
              if e in df_shift['emo_post'].values or e in df_shift['emo_disc'].values]

counts_mat = pd.crosstab(df_shift['emo_post'], df_shift['emo_disc'])
counts_mat = counts_mat.reindex(
    index   =[e for e in emos_order if e in counts_mat.index],
    columns =[e for e in emos_order if e in counts_mat.columns],
    fill_value=0
)
pct_mat = counts_mat.div(counts_mat.sum(axis=1), axis=0) * 100  # % per riga

# Heatmap
text_mat = [[f'{v:.0f}%' if v > 0 else '' for v in row] for row in pct_mat.values]
fig = go.Figure(go.Heatmap(
    z        = pct_mat.values,
    x        = list(pct_mat.columns),
    y        = list(pct_mat.index),
    colorscale='YlOrRd',
    text     = text_mat,
    texttemplate='%{text}',
    colorbar_title='%',
))
fig.update_layout(
    title      = 'Shift emotivo: emozione del post → emozione della discussione  (% per riga)',
    xaxis_title= 'Emozione discussione',
    yaxis_title= 'Emozione post',
    height     = 480,
)
fig.show()

In [44]:
print(pct_mat.round(1).to_string())

emo_disc     gioia  aspettativa  rabbia  disgusto  tristezza  sorpresa  paura  fiducia
emo_post                                                                              
gioia         51.9          1.9     0.0      15.4        3.8      11.5    5.8      9.6
aspettativa   16.7         16.7     8.3       0.0        8.3       0.0    8.3     41.7
rabbia        12.5         12.5    12.5      50.0        0.0       0.0   12.5      0.0
disgusto      22.7          4.5     0.0      40.9        4.5      13.6    0.0     13.6
tristezza     31.6          5.3     5.3      26.3       26.3       5.3    0.0      0.0
sorpresa      16.7         20.0     3.3      20.0       10.0      16.7    6.7      6.7
paura          6.7          0.0     6.7      53.3        0.0       6.7   13.3     13.3
fiducia       33.3         27.8     0.0      11.1        0.0       0.0   11.1     16.7


Ora analiziamo lo shift di **polarità** tra post e discussione, considerando le emozioni positive e negative.

In [46]:
# Matrice polarità positiva/ negativa
pol_mat = pd.crosstab(df_shift['pol_post'], df_shift['pol_disc'], margins=False)
pol_pct = pol_mat.div(pol_mat.sum(axis=1), axis=0) * 100
pol_order = [p for p in ['positiva','negativa','neutral'] if p in pol_pct.index]
pol_pct = pol_pct.reindex(
    index  =[p for p in pol_order if p in pol_pct.index],
    columns=[p for p in pol_order if p in pol_pct.columns],
    fill_value=0
)

print('=== SHIFT DI POLARITÀ (post → discussione) ===')
print(pol_pct.round(1).to_string())

=== SHIFT DI POLARITÀ (post → discussione) ===
pol_disc  positiva  negativa
pol_post                    
positiva      71.4      28.6
negativa      38.9      61.1


In [47]:
# Top-3 shift per ciascuna emozione del post
print('=== TOP SHIFT PER EMOZIONE DEL POST ===')
for emo in BASIC_EMOTIONS + ['neutrale']:
    sub = df_shift[df_shift['emo_post'] == emo]
    if len(sub) == 0:
        continue
    top  = sub['emo_disc'].value_counts()
    same = (sub['emo_disc'] == emo).sum()
    diff = len(sub) - same
    print(f'\n{emo.upper()}  ({len(sub)} post):')
    print(f'  stessa emozione: {same}/{len(sub)} ({same/len(sub)*100:.0f}%)')
    print(f'  shift:           {diff}/{len(sub)} ({diff/len(sub)*100:.0f}%)')
    top3 = '  '.join(f'{e}({cnt/len(sub)*100:.0f}%)' for e, cnt in top.head(3).items())
    print(f'  distribuzione → {top3}')

=== TOP SHIFT PER EMOZIONE DEL POST ===

GIOIA  (52 post):
  stessa emozione: 27/52 (52%)
  shift:           25/52 (48%)
  distribuzione → gioia(52%)  disgusto(15%)  sorpresa(12%)

ASPETTATIVA  (12 post):
  stessa emozione: 2/12 (17%)
  shift:           10/12 (83%)
  distribuzione → fiducia(42%)  aspettativa(17%)  gioia(17%)

RABBIA  (16 post):
  stessa emozione: 2/16 (12%)
  shift:           14/16 (88%)
  distribuzione → disgusto(50%)  aspettativa(12%)  rabbia(12%)

DISGUSTO  (22 post):
  stessa emozione: 9/22 (41%)
  shift:           13/22 (59%)
  distribuzione → disgusto(41%)  gioia(23%)  fiducia(14%)

TRISTEZZA  (19 post):
  stessa emozione: 5/19 (26%)
  shift:           14/19 (74%)
  distribuzione → gioia(32%)  disgusto(26%)  tristezza(26%)

SORPRESA  (30 post):
  stessa emozione: 5/30 (17%)
  shift:           25/30 (83%)
  distribuzione → disgusto(20%)  aspettativa(20%)  sorpresa(17%)

PAURA  (15 post):
  stessa emozione: 2/15 (13%)
  shift:           13/15 (87%)
  distribuzione 

In [48]:
# Heatmap polarità
fig2 = go.Figure(go.Heatmap(
    z         = pol_pct.values,
    x         = list(pol_pct.columns),
    y         = list(pol_pct.index),
    colorscale = 'Blues',
    text      = [[f'{v:.0f}%' for v in row] for row in pol_pct.values],
    texttemplate='%{text}',
    colorbar_title='%',
))
fig2.update_layout(
    title      = 'Shift di polarità: post → discussione  (% per riga)',
    xaxis_title= 'Polarità discussione',
    yaxis_title= 'Polarità post',
    height     = 350,
)
fig2.show()